# Home Credit — tổng quan dữ liệu và baseline

Mục tiêu: kiểm tra dữ liệu, dành riêng tập test, khám phá tập phát triển và tạo baseline có cross-validation.
Đây là notebook khởi đầu, chưa bao gồm toàn bộ thí nghiệm trong đề.

**Chuẩn bị:**
1. Tải `application_train.csv` và `HomeCredit_columns_description.csv` từ [Kaggle](https://www.kaggle.com/competitions/home-credit-default-risk/data).
2. Đặt hai tệp vào `data/raw/home_credit/` trong repo.
3. Cài thư viện: `pip install -r requirements.txt` ở gốc repo.
4. Chạy từng ô từ trên xuống, đọc kết quả trước khi chuyển ô tiếp theo.

Chỉ dùng bảng application_train. TARGET là nhãn; SK_ID_CURR là mã hồ sơ, không đưa vào đầu vào mô hình.
Không dùng application_test.csv làm test đánh giá vì tệp đó không cung cấp nhãn.


Thư viện cần dùng được khai báo trong `requirements.txt` ở gốc repo.


## 1. Cấu hình
Notebook tự tìm gốc repo (thư mục chứa `requirements.txt` và `src/`), nên không phụ thuộc
thư mục đang mở notebook. Nếu thiếu tệp, ô đọc dữ liệu sẽ dừng và in vị trí cần đặt tệp.


In [ ]:
from pathlib import Path
import json
import platform
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / 'requirements.txt').is_file() and (folder / 'src').is_dir():
            return folder
    raise FileNotFoundError(f'Không tìm thấy gốc repo khi đi lên từ {start}')

ROOT = find_project_root()
DATA_DIR = ROOT / 'data' / 'raw' / 'home_credit'
SPLIT_DIR = ROOT / 'data' / 'splits'
FIG_DIR = ROOT / 'results' / 'tabular' / 'figures'
TABLE_DIR = ROOT / 'results' / 'tabular' / 'tables'
for folder in (SPLIT_DIR, FIG_DIR, TABLE_DIR):
    folder.mkdir(parents=True, exist_ok=True)
SEED = 42
TEST_SIZE = 0.20
print('CSV cần đặt tại:', DATA_DIR.resolve())
print('Gốc repo:', ROOT)
print('Hình lưu tại:', FIG_DIR, '| Bảng lưu tại:', TABLE_DIR)
versions = {'python': platform.python_version(), 'pandas': pd.__version__,
            'numpy': np.__version__, 'scikit-learn': sklearn.__version__}
print(versions)
(TABLE_DIR / 'environment.json').write_text(json.dumps(versions, indent=2), encoding='utf-8')

In [ ]:
csv_path = DATA_DIR / 'application_train.csv'
if not csv_path.is_file():
    raise FileNotFoundError(f'Hãy tải application_train.csv và đặt tại: {csv_path.resolve()}')
df = pd.read_csv(csv_path)
print('Số dòng, số cột:', df.shape)
display(df.head())

description_path = DATA_DIR / 'HomeCredit_columns_description.csv'
if description_path.is_file():
    # Tệp mô tả của Kaggle dùng bảng mã Windows-1252 (ví dụ dấu '…'), không phải UTF-8.
    description = pd.read_csv(description_path, encoding='cp1252', index_col=0)
    display(description.head(10))
else:
    print('Chưa có bảng mô tả cột. Hãy tải thêm để diễn giải ý nghĩa thuộc tính.')

## 2. Kiểm tra dữ liệu gốc
Phần này kiểm tra cấu trúc và chất lượng đầu vào. Chưa xóa dòng thiếu, chưa điền khuyết,
chưa xóa ngoại lai và chưa chọn đặc trưng theo nhãn. Báo cáo tỷ lệ thiếu theo từng cột.
Các kiểm tra dưới đây phải đạt trước khi tiếp tục.

In [ ]:
assert {'SK_ID_CURR', 'TARGET'}.issubset(df.columns), 'Thiếu ID hoặc nhãn.'
assert df['SK_ID_CURR'].notna().all(), 'Có ID bị thiếu; cần kiểm tra.'
assert df['SK_ID_CURR'].is_unique, 'Có ID trùng; cần xử lý nguyên nhân trước khi chia tập.'
assert df['TARGET'].notna().all(), 'Có nhãn thiếu.'
assert set(df['TARGET'].unique()) == {0, 1}, 'Nhãn không đúng hai lớp 0/1.'

features = df.drop(columns=['SK_ID_CURR', 'TARGET'])
audit = pd.DataFrame({
    'dtype': features.dtypes.astype(str),
    'missing_count': features.isna().sum(),
    'missing_pct': features.isna().mean().mul(100),
    'unique_non_missing': features.nunique(dropna=True),
}).sort_values('missing_pct', ascending=False)
assert len(df) >= 10_000, 'Chưa đủ 10.000 dòng.'
assert features.shape[1] >= 10, 'Chưa đủ 10 thuộc tính đầu vào.'
assert (audit['missing_pct'] >= 5).any(), 'Không có cột thiếu từ 5% trở lên.'
assert len(features.select_dtypes(include='number').columns) > 0
assert len(features.select_dtypes(exclude='number').columns) > 0
display(audit.head(20))
audit.to_csv(TABLE_DIR / 'data_audit.csv', encoding='utf-8-sig')
print('Số dòng trùng toàn bộ:', int(df.duplicated().sum()))
print('Đã đạt các điều kiện kích thước, loại biến và tỷ lệ thiếu.')

## 3. Chia tập và lưu danh sách ID
Giữ 20% làm test cuối cùng; 80% là tập phát triển (development).
Trong tập phát triển, dùng 5-fold CV để huấn luyện và kiểm tra các cấu hình.
Các biểu đồ và lựa chọn tiếp theo chỉ dùng tập phát triển.
Danh sách ID được lưu để những notebook khác sử dụng đúng cách chia này.

In [ ]:
dev_idx, test_idx = train_test_split(
    df.index, test_size=TEST_SIZE, stratify=df['TARGET'], random_state=SEED
)
dev = df.loc[dev_idx].copy()
holdout = df.loc[test_idx].copy()
assert set(dev['SK_ID_CURR']).isdisjoint(set(holdout['SK_ID_CURR']))
split_ids = pd.concat([
    dev[['SK_ID_CURR']].assign(split='development'),
    holdout[['SK_ID_CURR']].assign(split='test'),
], ignore_index=True)
split_ids.to_csv(SPLIT_DIR / 'home_credit_split_ids.csv', index=False)
print('Development:', len(dev), '| Test dành riêng:', len(holdout))
display(dev['TARGET'].value_counts().sort_index().rename('count').to_frame().assign(
    percent=dev['TARGET'].value_counts(normalize=True).sort_index().mul(100)
))

## 4. Khám phá ban đầu trên development
Đọc mô tả cột trước khi giải thích. Mã số không nhất thiết là biến liên tục.
Notebook này mới cung cấp biểu đồ ban đầu; các kiểm định phân phối, MCAR,
phân tích tương quan đầy đủ và thí nghiệm có kiểm soát sẽ làm ở notebook EDA tiếp theo.

In [ ]:
dev_missing = dev.drop(columns=['SK_ID_CURR', 'TARGET']).isna().mean().mul(100)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
dev['TARGET'].value_counts().sort_index().plot.bar(ax=axes[0], rot=0)
axes[0].set(title='Phân bố nhãn trên development', xlabel='TARGET', ylabel='Số hồ sơ')
dev_missing.nlargest(15).sort_values().plot.barh(ax=axes[1])
axes[1].set(title='15 cột có tỷ lệ thiếu cao nhất', xlabel='% thiếu')
fig.tight_layout()
fig.savefig(FIG_DIR / 'labels_and_missing.png', dpi=160, bbox_inches='tight')
plt.show()

selected = [c for c in ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
                       'DAYS_BIRTH', 'OWN_CAR_AGE', 'EXT_SOURCE_1'] if c in dev]
display(dev[selected].describe(percentiles=[.01, .25, .5, .75, .99]).T)
if selected:
    dev[selected].hist(bins=40, figsize=(14, 8))
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'numeric_distributions.png', dpi=160, bbox_inches='tight')
    plt.show()

# Kiểm tra giả thuyết: thiếu tuổi xe có liên quan đến tình trạng sở hữu xe không?
if {'FLAG_OWN_CAR', 'OWN_CAR_AGE'}.issubset(dev.columns):
    display(pd.crosstab(dev['FLAG_OWN_CAR'], dev['OWN_CAR_AGE'].isna(), normalize='index'))

### Ghi nhận xét của bạn tại đây
- Lớp nào ít mẫu hơn? Accuracy có đủ để đánh giá không?
- Ba cột thiếu nhiều nhất là gì? Có thể giải thích nguyên nhân thiếu không?
- Biến nào lệch mạnh? Giá trị cực đoan có chắc là lỗi dữ liệu không?
- Dấu âm hoặc giá trị đặc biệt trong các cột DAYS_* có ý nghĩa gì trong tài liệu?

Chưa kết luận MCAR/MAR/MNAR chỉ từ các biểu đồ trên. Không tự động coi mọi giá trị lạ là ngoại lai cần xóa.

## 5. Pipeline baseline
Giữ các quy tắc đơn giản làm mốc: median + StandardScaler cho cột số,
mode + One-Hot cho cột chuỗi; Logistic Regression không tái cân bằng.
Đây là lựa chọn baseline, không phải kết luận phương pháp tốt nhất.
Mọi phép fit đều nằm bên trong CV. Mô hình Dummy luôn dự đoán lớp phổ biến được dùng làm mốc tối thiểu.

One-Hot bỏ một mức tham chiếu mỗi cột; giá trị chưa gặp được xử lý mà không gây lỗi.
Cột hoàn toàn thiếu trong một fold vẫn được giữ để đảm bảo cấu trúc nhất quán.

In [ ]:
X_dev = dev.drop(columns=['SK_ID_CURR', 'TARGET'])
y_dev = dev['TARGET'].astype(int)
numeric_cols = X_dev.select_dtypes(include='number').columns.tolist()
categorical_cols = X_dev.select_dtypes(exclude='number').columns.tolist()

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('scaler', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipe, numeric_cols),
    ('categorical', categorical_pipe, categorical_cols),
])
baseline = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=2000, solver='lbfgs', random_state=SEED)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {'f1_macro': 'f1_macro', 'recall_positive': 'recall',
           'roc_auc': 'roc_auc', 'average_precision': 'average_precision',
           'accuracy': 'accuracy'}
print('Số cột số:', len(numeric_cols), '| Số cột chuỗi:', len(categorical_cols))

## 6. Đánh giá baseline bằng 5-fold CV
Ô dưới có thể mất thời gian trên toàn bộ development; thời gian phụ thuộc máy.
Giữ cảnh báo hội tụ hiển thị. Nếu có ConvergenceWarning, cần khắc phục trước khi coi kết quả là hoàn chỉnh.
Không tối ưu cấu hình dựa trên test dành riêng.
Average precision (AP) là thước đo tóm tắt đường precision–recall; ở đây báo cáo đúng tên AP.

In [ ]:
cv_baseline = cross_validate(
    baseline, X_dev, y_dev, cv=cv, scoring=scoring, n_jobs=1, error_score='raise'
)
cv_dummy = cross_validate(
    DummyClassifier(strategy='most_frequent'), np.zeros((len(y_dev), 1)), y_dev,
    cv=cv, scoring=scoring, n_jobs=1, error_score='raise'
)
records = []
for name, scores in [('dummy', cv_dummy), ('logistic_baseline', cv_baseline)]:
    for metric in scoring:
        values = scores['test_' + metric]
        records.append({'model': name, 'metric': metric,
                        'mean': values.mean(), 'std': values.std(ddof=1)})
summary = pd.DataFrame(records)
display(summary)
summary.to_csv(TABLE_DIR / 'baseline_cv_summary.csv', index=False)
pd.DataFrame(cv_baseline).to_csv(TABLE_DIR / 'baseline_cv_folds.csv', index=False)
pd.DataFrame(cv_dummy).to_csv(TABLE_DIR / 'dummy_cv_folds.csv', index=False)
config = {'seed': SEED, 'test_size': TEST_SIZE, 'cv_folds': 5,
          'development_rows': len(dev), 'reserved_test_rows': len(holdout),
          'sampling': 'none', 'numeric_columns': numeric_cols,
          'categorical_columns': categorical_cols,
          'class_weight': None, 'resampling': 'none',
          'note': 'CV only; reserved test not evaluated'}
(TABLE_DIR / 'baseline_config.json').write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8'
)

## 7. Kết thúc buổi đầu

Bạn đã có: dữ liệu gốc, bảng audit, split ID cố định, biểu đồ ban đầu và kết quả baseline CV.
Hãy viết nhận xét từ kết quả thật, không điền trước kết quả mong muốn.

Thứ tự tiếp theo:
1. Hoàn thiện EDA: kiểm định phân phối, tương quan, phân tích cơ chế thiếu với giả định rõ ràng.
2. So sánh điền thiếu trên các ô quan sát được rồi che nhân tạo, giữ cùng mask giữa phương pháp.
3. Thử ngoại lai, scaling, encoding và lựa chọn đặc trưng, mỗi lần thay một nhóm yếu tố.
4. Cố định pipeline cuối rồi mới đánh giá test đã dành riêng.

Nguồn kỹ thuật: [Pipeline và chống leakage](https://scikit-learn.org/stable/common_pitfalls.html),
[SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html),
[StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html).
Đối chiếu toàn bộ bài nộp với [đề HCMUS](https://hcmus-cs-ta.github.io/lab-book/data-mining/projects/preprocessing/).